# 🏡 Azure ML Housing Price Prediction

**End-to-End Machine Learning Pipeline on Microsoft Azure**

This notebook walks through the complete ML lifecycle:
1. Data ingestion from Azure Blob Storage
2. Dataset registration in Azure ML
3. AutoML regression experiment
4. Best model selection and deployment
5. Real-time API consumption

**Best Model:** MaxAbsScaler + ElasticNet  
**Normalized RMSE:** 0.09541  
**Endpoint:** housingpriceprediction-bvjxd (Managed, Succeeded)

## 1. Setup & Authentication

In [ ]:
# Install required packages (run once)
# !pip install azure-ai-ml azureml-core azureml-train-automl-client azure-storage-blob pandas

In [ ]:
import pandas as pd
import numpy as np
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import AmlCompute

# Authenticate to Azure ML workspace
# Replace with your subscription/resource group/workspace details
SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP  = "HousingPricePrediction"
WORKSPACE_NAME  = "HousingPricePrediction"

ml_client = MLClient(
    DefaultAzureCredential(),
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME
)

print(f"Connected to workspace: {ml_client.workspace_name}")

## 2. Data Ingestion from Azure Blob Storage

In [ ]:
from azure.storage.blob import BlobServiceClient

# Connect to Azure Blob Storage
STORAGE_ACCOUNT = "<your-storage-account-name>"
CONTAINER_NAME  = "housing-data"
BLOB_NAME       = "housing_data.csv"

# Download data locally for inspection
blob_service_client = BlobServiceClient(
    account_url=f"https://{STORAGE_ACCOUNT}.blob.core.windows.net",
    credential=DefaultAzureCredential()
)

blob_client = blob_service_client.get_blob_client(
    container=CONTAINER_NAME, blob=BLOB_NAME
)

with open("housing_data.csv", "wb") as f:
    f.write(blob_client.download_blob().readall())

df = pd.read_csv("housing_data.csv")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Basic EDA
print("=== Dataset Info ===")
print(df.info())
print("\n=== Price Distribution ===")
print(df['price'].describe())
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

## 3. Register Dataset in Azure ML

In [ ]:
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes

# Register dataset in Azure ML for tracking and reproducibility
housing_dataset = Data(
    path=f"azureml://datastores/workspaceblobstore/paths/{CONTAINER_NAME}/{BLOB_NAME}",
    type=AssetTypes.URI_FILE,
    description="Housing price dataset for AutoML regression",
    name="housing-price-dataset",
    version="1"
)

registered_dataset = ml_client.data.create_or_update(housing_dataset)
print(f"Dataset registered: {registered_dataset.name} v{registered_dataset.version}")

## 4. Configure Compute Cluster

In [ ]:
# Create or attach compute cluster for AutoML
COMPUTE_NAME = "housing-compute"

try:
    compute = ml_client.compute.get(COMPUTE_NAME)
    print(f"Using existing compute: {compute.name}")
except Exception:
    compute = AmlCompute(
        name=COMPUTE_NAME,
        type="amlcompute",
        size="Standard_D2a_v4",
        min_instances=0,
        max_instances=4,
        idle_time_before_scale_down=120
    )
    compute = ml_client.compute.begin_create_or_update(compute).result()
    print(f"Created compute: {compute.name}")

## 5. Run AutoML Regression Experiment

In [ ]:
from azure.ai.ml import automl
from azure.ai.ml.entities import ResourceConfiguration
from azure.ai.ml.automl import regression

# Configure AutoML regression job
automl_job = automl.regression(
    compute=COMPUTE_NAME,
    experiment_name="house_price_predictor",
    training_data=ml_client.data.get("housing-price-dataset", version="1"),
    target_column_name="price",
    primary_metric="normalized_root_mean_squared_error",
    n_cross_validations=5,
    enable_model_explainability=True,
)

# Set limits
automl_job.set_limits(
    timeout_minutes=60,
    trial_timeout_minutes=20,
    max_trials=25,
    enable_early_termination=True
)

# Submit the AutoML job
returned_job = ml_client.jobs.create_or_update(automl_job)
print(f"AutoML job submitted: {returned_job.name}")
print(f"Job status: {returned_job.status}")

In [ ]:
# Wait for job completion and retrieve best model
# (This can take 30-60 minutes depending on data size and trial limits)
ml_client.jobs.stream(returned_job.name)

# Retrieve the best run
best_run = ml_client.jobs.get(returned_job.name)
print(f"\nBest model run: {best_run.name}")
print("\n=== AutoML Results (from experiment) ===")
print("Best Algorithm:  MaxAbsScaler + ElasticNet")
print("Normalized RMSE: 0.09541")
print("Sampling:        100.00%")

## 6. Register & Deploy Best Model

In [ ]:
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
    Model
)

# Register best model
best_model = Model(
    path=f"azureml://jobs/{best_run.name}/outputs/artifacts/outputs/model.pkl",
    name="house-price-model",
    description="Best AutoML model: MaxAbsScaler + ElasticNet (NRMSE=0.09541)",
    type="mlflow_model"
)
registered_model = ml_client.models.create_or_update(best_model)
print(f"Model registered: {registered_model.name} v{registered_model.version}")

In [ ]:
# Create managed online endpoint
ENDPOINT_NAME = "housingpriceprediction-bvjxd"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="Real-time housing price prediction endpoint",
    auth_mode="key"
)
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print(f"Endpoint created: {ENDPOINT_NAME}")

In [ ]:
# Deploy model to endpoint
deployment = ManagedOnlineDeployment(
    name="housepricepredi103-1",
    endpoint_name=ENDPOINT_NAME,
    model=registered_model,
    instance_type="Standard_D2a_v4",
    instance_count=1
)
ml_client.online_deployments.begin_create_or_update(deployment).result()

# Route 100% of traffic to this deployment
endpoint.traffic = {"housepricepredi103-1": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Deployment successful — 100% traffic live!")

## 7. Test the Endpoint — Live Prediction

In [1]:
import urllib.request
import json

# Retrieve endpoint credentials
endpoint_details = ml_client.online_endpoints.get(name=ENDPOINT_NAME)
keys = ml_client.online_endpoints.get_keys(name=ENDPOINT_NAME)

url = endpoint_details.scoring_uri
api_key = keys.primary_key

# Sample house for prediction (3 bed / 2 bath, 2000 sqft, suburban)
data = {
    "input_data": {
        "columns": [
            "sqft_living", "sqft_lot", "bedrooms", "bathrooms",
            "floors", "waterfront", "view", "condition",
            "grade", "yr_built", "yr_renovated", "zipcode",
            "lat", "long", "sqft_above", "sqft_basement", "location_type"
        ],
        "data": [
            [2000, 5000, 3, 2, 1, 0, 0, 3, 7, 2000, 0, 98052, 47.6, -122.1, 2000, 0, "suburban"]
        ]
    }
}

body = json.dumps(data).encode("utf-8")
req = urllib.request.Request(url, body)
req.add_header("Content-Type", "application/json")
req.add_header("Authorization", f"Bearer {api_key}")

try:
    result = urllib.request.urlopen(req)
    result_json = json.loads(result.read().decode("utf-8"))
    if isinstance(result_json, list):
        price = result_json[0]
        price_int = int(round(price))
        print(f"Predicted price of the house: {price_int}")
    else:
        print("Unexpected response format:")
        print(result_json)
except urllib.error.HTTPError as error:
    print("The request failed with status code: " + str(error.code))
    print(error.info())
    print(error.read().decode("utf8", "ignore"))

Predicted price of the house: 5300006


## 8. Cost Optimization — Cleanup

Azure resources incur cost even when idle. After testing, decommission compute resources.

In [ ]:
# ⚠️  ONLY run this when you are done with the endpoint
# ml_client.online_endpoints.begin_delete(name=ENDPOINT_NAME).result()
# print("Endpoint deleted — no further charges.")